In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import Database CRUD Module
from CRUD_Python_Module_Final import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "M0ng0L34Rn"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(style = {'display': 'flex'}, 
             children = [
            html.A(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style = {'width' : '100px'}), href="https://www.snhu.edu/", target = "_blank"),
            
            html.B(html.H1('CS-340 Dashboard - Michael Michel'))
             ]),
    html.Hr(),
    html.Div(
        
    ),
    html.Div(style = {'display': 'flex'},
             children = [
        html.H3("Filter by Rescue Type:"),
        dcc.Dropdown(["None", "Water Rescue", "Mountain/Wilderness Rescue", "Disaster Rescue/Individual Rescue"], "None", 
                     id="animal-dropdown", style = {'width' : '70%', 'margin-top' : '6px'}),
        ]
    ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        row_selectable = "single",
        selected_rows = [0], #Default Row
        page_size = 30,
        style_table = {"height" : "400px", "overflowY" : "auto"},
        sort_action = "native",
        # filter_action = "native" - Not as user intuitive but super useful
        style_cell = {
            "textAlign": "Center"
        },
        style_data = {
            "color": "black",
            "backgroundColor": "white"
        },
        style_data_conditional = [
            {
                "if": {"row_index": "odd"},
                "backgroundColor": "rgb(200, 200, 200)",
            }
        ],
        style_header = {
            "backgroundColor": "rgb(180, 180, 180)",
            "color": "black",
            "fontWeight": "bold",
            "text-transform": "capitalize"
        }
        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

#Dropdown Filter
@app.callback(
    Output("datatable-id", "data"),
    Input("animal-dropdown", "value")
)
def update_output(value):
    #Default Read
    df = pd.DataFrame.from_records(db.read({}))
   
    if value == "Water Rescue":
        df = pd.DataFrame.from_records(db.read({"$and" : [
                                                    {"$or": [
                                                        {'breed' : {'$regex': 'Labrador Retriever Mix'}},
                                                        {'breed' : {'$regex': 'Chesapeake Bay Retriever'}},
                                                        {'breed' : {'$regex': 'Newfoundland'}}
                                                    ]},
                                                    {'sex_upon_outcome' : 'Intact Female'},
                                                    {"$and" : [
                                                        {"age_upon_outcome_in_weeks" : {"$lt" : 156}},
                                                        {"age_upon_outcome_in_weeks" : {"$gt" : 26}}]
                                                    }]
                                               }))
    elif value == "Mountain/Wilderness Rescue":
        df = pd.DataFrame.from_records(db.read({"$and" : [
                                                    {"$or": [
                                                        {'breed' : {'$regex': 'German Shepherd'}},
                                                        {'breed' : {'$regex': 'Alaskan Malamute'}},
                                                        {'breed' : {'$regex': 'Old English Sheepdog'}},
                                                        {'breed' : {'$regex': 'Siberian Husky'}},
                                                        {'breed' : {'$regex': 'Rottweiler'}}
                                                    ]},
                                                    {'sex_upon_outcome' : 'Intact Male'},
                                                    {"$and" : [
                                                        {"age_upon_outcome_in_weeks" : {"$lt" : 156}},
                                                        {"age_upon_outcome_in_weeks" : {"$gt" : 26}}]
                                                    }]
                                                }))
    elif value == "Disaster Rescue/Individual Rescue":
        df = pd.DataFrame.from_records(db.read({"$and" : [
                                                    {"$or": [
                                                        {'breed' : {'$regex': 'German Shepherd'}},
                                                        {'breed' : {'$regex': 'Doberman Pinscher'}},
                                                        {'breed' : {'$regex': 'Golden Retriever'}},
                                                        {'breed' : {'$regex': 'Rottweiler'}},
                                                        {'breed' : {'$regex': 'Bloodhound'}}
                                                    ]},
                                                    {'sex_upon_outcome' : 'Intact Male'},
                                                    {"$and" : [
                                                        {"age_upon_outcome_in_weeks" : {"$lt" : 300}},
                                                        {"age_upon_outcome_in_weeks" : {"$gt" : 20}}]
                                                    }]
                                               }))
        
    df.drop(columns=["_id"], inplace = True)
    
    return df.to_dict("records")

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    df = pd.DataFrame.from_records(viewData)
    return [
        dcc.Graph(            
           figure = px.pie(df, names='breed', title='Preferred Animals', hole = .4)
        )    
    ]

#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode='jupyterlab', port='8069') 

C:\Users\monta\AppData\Local\Programs\Python\Python314\Lib\site-packages\dash\dash.py:642: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



Dash app running on http://127.0.0.1:8069/


<IPython.core.display.Javascript object>